# Data Integration / Joining

This notebook integrates the Instacart relational datasets into a customer-level analytical dataset for churn prediction.

The integration preserves customer order history, purchase frequency, product-category information, and reorder behavior while avoiding unnecessary expansion of the final customer-level dataset.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## Load Core Reference Tables

In [3]:
orders = pd.read_csv(
    DATA_DIR / "orders.csv"
)

products = pd.read_csv(
    DATA_DIR / "products.csv"
)

aisles = pd.read_csv(
    DATA_DIR / "aisles.csv"
)

departments = pd.read_csv(
    DATA_DIR / "departments.csv"
)

print("Orders:", orders.shape)
print("Products:", products.shape)
print("Aisles:", aisles.shape)
print("Departments:", departments.shape)

Orders: (3421083, 7)
Products: (49688, 4)
Aisles: (134, 2)
Departments: (21, 2)


In [4]:
product_catalog = (
    products
    .merge(
        aisles,
        on="aisle_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        departments,
        on="department_id",
        how="left",
        validate="many_to_one"
    )
)

product_catalog.shape

(49688, 6)

## Integrate Order-Level Information

The prior and train order-product tables are combined at the transaction level.

The resulting transaction data will retain the customer relationship through `order_id` while preserving product and reorder information for subsequent customer-level aggregation.

In [5]:
ORDER_COLUMNS = [
    "order_id",
    "product_id",
    "add_to_cart_order",
    "reordered"
]

prior_path = DATA_DIR / "order_products__prior.csv"
train_path = DATA_DIR / "order_products__train.csv"

In [6]:
orders_lookup = orders[
    ["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day"]
].copy()

orders_lookup = orders_lookup.set_index("order_id")

In [7]:
customer_aggregates = []

for chunk in pd.read_csv(
    prior_path,
    usecols=ORDER_COLUMNS,
    chunksize=200_000
):
    chunk = chunk.join(orders_lookup, on="order_id")

    aggregated = (
        chunk
        .groupby("user_id")
        .agg(
            prior_order_items=("product_id", "count"),
            prior_unique_products=("product_id", "nunique"),
            prior_reordered_items=("reordered", "sum"),
            prior_orders=("order_id", "nunique")
        )
        .reset_index()
    )

    customer_aggregates.append(aggregated)

In [8]:
customer_prior = (
    pd.concat(customer_aggregates)
    .groupby("user_id", as_index=False)
    .sum()
)

customer_prior.head()

,user_id,prior_order_items,prior_unique_products,prior_reordered_items,prior_orders
0,1,59,57,41,10
1,2,195,192,93,14
2,3,88,88,55,12
3,4,18,18,1,5
4,5,37,37,14,4


## Combine Customer-Level Transaction Data

Prior-order and train-order aggregates are combined with the customer order history.

The resulting dataset contains one record per customer and serves as the foundation for subsequent churn feature engineering.

In [9]:
customer_orders = (
    orders
    .sort_values(["user_id", "order_number"])
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "nunique"),
        max_order_number=("order_number", "max"),
        avg_days_between_orders=("days_since_prior_order", "mean")
    )
    .reset_index()
)

customer_orders.head()

,user_id,total_orders,max_order_number,avg_days_between_orders
0,1,11,11,19.000000
1,2,15,15,16.285714
2,3,13,13,12.000000
3,4,6,6,17.000000
4,5,5,5,11.500000


In [10]:
latest_orders = (
    orders
    .sort_values(["user_id", "order_number"])
    .groupby("user_id")
    .tail(1)
    [
        [
            "user_id",
            "order_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order"
        ]
    ]
    .rename(
        columns={
            "order_number": "latest_order_number",
            "order_dow": "latest_order_dow",
            "order_hour_of_day": "latest_order_hour",
            "days_since_prior_order": "latest_days_since_prior_order"
        }
    )
)

latest_orders.head()

,user_id,order_id,latest_order_number,latest_order_dow,latest_order_hour,latest_days_since_prior_order
10,1,1187899,11,4,8,14.0
25,2,1492625,15,1,11,30.0
38,3,2774568,13,5,15,11.0
44,4,329954,6,3,12,30.0
49,5,2196797,5,0,11,6.0


In [11]:
customer_data = (
    customer_orders
    .merge(customer_prior, on="user_id", how="left")
    .merge(latest_orders, on="user_id", how="left")
)

customer_data.shape

(206209, 13)

In [12]:
print("Customers in orders:", orders["user_id"].nunique())
print("Customers in final dataset:", customer_data["user_id"].nunique())
print("Final rows:", len(customer_data))
print("Duplicate user_id:", customer_data["user_id"].duplicated().sum())

Customers in orders: 206209
Customers in final dataset: 206209
Final rows: 206209
Duplicate user_id: 0


In [13]:
customer_data.head()

,user_id,total_orders,max_order_number,avg_days_between_orders,prior_order_items,prior_unique_products,prior_reordered_items,prior_orders,order_id,latest_order_number,latest_order_dow,latest_order_hour,latest_days_since_prior_order
0,1,11,11,19.000000,59,57,41,10,1187899,11,4,8,14.0
1,2,15,15,16.285714,195,192,93,14,1492625,15,1,11,30.0
2,3,13,13,12.000000,88,88,55,12,2774568,13,5,15,11.0
3,4,6,6,17.000000,18,18,1,5,329954,6,3,12,30.0
4,5,5,5,11.500000,37,37,14,4,2196797,5,0,11,6.0


In [15]:
prior_count_columns = [
    "prior_order_items",
    "prior_unique_products",
    "prior_reordered_items",
    "prior_orders"
]

customer_data[prior_count_columns] = (
    customer_data[prior_count_columns]
    .fillna(0)
)

In [16]:
print("Shape:", customer_data.shape)
print("Unique customers:", customer_data["user_id"].nunique())
print("Duplicate user_id:", customer_data["user_id"].duplicated().sum())
print("Missing values:")
print(customer_data.isna().sum())

Shape: (206209, 13)
Unique customers: 206209
Duplicate user_id: 0
Missing values:
user_id                          0
total_orders                     0
max_order_number                 0
avg_days_between_orders          0
prior_order_items                0
prior_unique_products            0
prior_reordered_items            0
prior_orders                     0
order_id                         0
latest_order_number              0
latest_order_dow                 0
latest_order_hour                0
latest_days_since_prior_order    0
dtype: int64


## Export Prepared Customer Dataset

The integrated customer-level dataset is exported to the interim data directory.

This dataset contains one record per customer and represents the validated foundation for subsequent churn feature engineering.

In [17]:
output_path = INTERIM_DIR / "customer_integrated.csv"

customer_data.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(f"Rows: {len(customer_data):,}")
print(f"Columns: {len(customer_data.columns)}")

Saved: ..\data\interim\customer_integrated.csv
Rows: 206,209
Columns: 13


In [18]:
exported_data = pd.read_csv(output_path)

print("Exported shape:", exported_data.shape)
print("Exported duplicate user_id:",
      exported_data["user_id"].duplicated().sum())

Exported shape: (206209, 13)
Exported duplicate user_id: 0
